# Advanced: representation and invariance audit

A classifier can produce the right label for the wrong reason. It can also preserve nuisance information that never appears in its final prediction. This notebook looks inside the released TorchSig Models v1.0 XCiT and asks two questions at several network depths:

1. **Does the representation preserve waveform-family information under controlled impairments?**
2. **Can a linear probe still recover which impairment was applied?**

We'll create paired views of the same underlying signals, capture intermediate token embeddings with forward hooks, measure cosine drift, and fit identity-safe linear probes. This is an audit of one small controlled dataset—not a claim that the model is globally invariant.

## 1. Environment and released checkpoint

This notebook adds scikit-learn for probes and PCA. The XCiT checkpoint downloads from the official v1.0.0 release and is cached locally. A GPU helps, but the default audit is small enough to run on CPU.

In [ ]:
import importlib.util
import subprocess
import sys

requirements = []
if importlib.util.find_spec('torchsig_models') is None:
    requirements.append('git+https://github.com/TorchDSP/torchsig-models.git@v1.0.0')
if importlib.util.find_spec('matplotlib') is None:
    requirements.append('matplotlib>=3.7')
if importlib.util.find_spec('sklearn') is None:
    requirements.append('scikit-learn>=1.3')
if requirements:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *requirements])
    print('Installation complete. Restart the runtime if an import still fails.')
else:
    print('All dependencies are ready.')

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from torchsig.signals.signal_types import Signal
from torchsig.transforms.transforms import AWGN, CarrierPhaseOffset
from torchsig_models.models import XCiTClassifier

CHECKPOINT = Path('xcit_narrowband_v1.0.0.ckpt')
CHECKPOINT_URL = (
    'https://github.com/TorchDSP/torchsig-models/releases/download/'
    'v1.0.0/xcit_narrowband_v1.0.0.ckpt'
)
if not CHECKPOINT.exists():
    urlretrieve(CHECKPOINT_URL, CHECKPOINT)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = XCiTClassifier.load_from_checkpoint(CHECKPOINT, map_location=DEVICE)
model.to(DEVICE).eval()
print(f'Loaded XCiT on {DEVICE}; transformer blocks={len(model.model.backbone.blocks)}')

## 2. Construct paired counterfactual views

We generate three deliberately different waveform families: tones, rising linear chirps, and rectangular QPSK. Each underlying identity receives four views:

- clean;
- a 90-degree TorchSig `CarrierPhaseOffset`;
- a constant frequency offset of 0.08 cycles/sample;
- TorchSig `AWGN` at 0 dB SNR.

The phase and noise operations use TorchSig. TorchSig 2.1 does not expose an exact constant-offset transform, so the frequency translation remains explicit complex mixing. Crucially, all four views retain the same `identity_id`; this lets us split identities—not rows—and avoid leaking paired examples between probe train and test sets.

In [ ]:
SEED = 2026
NUM_SAMPLES = 4096
IDENTITIES_PER_CLASS = 18
CLASS_NAMES = ['tone', 'linear-chirp', 'qpsk']
VIEW_NAMES = ['clean', 'phase+90', 'frequency+0.08', 'snr=0dB']
rng = np.random.default_rng(SEED)
n = np.arange(NUM_SAMPLES)

def unit_power(iq):
    return np.asarray(iq / np.sqrt(np.mean(np.abs(iq) ** 2)), dtype=np.complex64)

base_iq, class_id = [], []
for _ in range(IDENTITIES_PER_CLASS):
    frequency = rng.uniform(-0.30, 0.30)
    phase = rng.uniform(0, 2 * np.pi)
    base_iq.append(unit_power(np.exp(1j * (2 * np.pi * frequency * n + phase))))
    class_id.append(0)
for _ in range(IDENTITIES_PER_CLASS):
    start = rng.uniform(-0.35, -0.10)
    end = rng.uniform(0.10, 0.35)
    slope = (end - start) / (NUM_SAMPLES - 1)
    phase = 2 * np.pi * (start * n + 0.5 * slope * n**2)
    base_iq.append(unit_power(np.exp(1j * (phase + rng.uniform(0, 2 * np.pi)))))
    class_id.append(1)
for _ in range(IDENTITIES_PER_CLASS):
    symbol_indices = rng.integers(0, 4, NUM_SAMPLES // 8)
    symbols = np.exp(1j * (np.pi / 4 + symbol_indices * np.pi / 2))
    qpsk = np.repeat(symbols, 8) * np.exp(1j * rng.uniform(0, 2 * np.pi))
    base_iq.append(unit_power(qpsk))
    class_id.append(2)
base_iq = np.stack(base_iq)
class_id = np.asarray(class_id)
identity_id = np.arange(len(base_iq))
print(f'Base identities: {len(base_iq)}; shape={base_iq.shape}')

In [ ]:
def make_view(iq, view_index, identity):
    if view_index == 0:
        return iq.copy()
    if view_index == 1:
        return CarrierPhaseOffset(
            phase_offset_range=(np.pi / 2, np.pi / 2), seed=SEED + identity
        )(Signal(data=iq.copy())).data
    if view_index == 2:
        return np.asarray(iq * np.exp(2j * np.pi * 0.08 * n), dtype=np.complex64)
    if view_index == 3:
        return AWGN(
            noise_power_db=0.0, seed=SEED + identity
        )(Signal(data=iq.copy())).data
    raise ValueError(view_index)

all_iq, all_class, all_view, all_identity = [], [], [], []
for view_index in range(len(VIEW_NAMES)):
    for identity, (iq, label) in enumerate(zip(base_iq, class_id)):
        all_iq.append(make_view(iq, view_index, identity))
        all_class.append(label)
        all_view.append(view_index)
        all_identity.append(identity)
all_iq = np.stack(all_iq).astype(np.complex64)
all_class = np.asarray(all_class)
all_view = np.asarray(all_view)
all_identity = np.asarray(all_identity)
print(f'Paired audit set: {len(all_iq)} examples; shape={all_iq.shape}')

In [ ]:
identity_to_show = IDENTITIES_PER_CLASS  # First chirp identity.
indices = np.flatnonzero(all_identity == identity_to_show)
fig, axes = plt.subplots(1, len(indices), figsize=(14, 3.2), constrained_layout=True)
for axis, index in zip(axes, indices):
    axis.specgram(all_iq[index], NFFT=256, Fs=1.0, noverlap=192, cmap='magma')
    axis.set_title(VIEW_NAMES[all_view[index]])
    axis.set_xlabel('Normalized time')
axes[0].set_ylabel('Normalized frequency')
fig.suptitle('Four paired views of one underlying chirp')
plt.show()

## 3. Capture representations across depth

Forward hooks observe module outputs without modifying the model. We sample four cross-covariance transformer blocks and the final normalized classification token. Block outputs contain one vector per token, so we mean-pool tokens into one embedding per example. For the final normalization output, we retain the classification token at position zero.

Hooks are powerful but easy to misuse: always remove them when extraction finishes, and verify tensor semantics for the exact architecture being audited.

In [ ]:
blocks = model.model.backbone.blocks
block_indices = sorted(set([0, len(blocks) // 3, 2 * len(blocks) // 3, len(blocks) - 1]))
layer_modules = {f'block-{index:02d}': blocks[index] for index in block_indices}
layer_modules['final-cls'] = model.model.backbone.norm
captured = {name: [] for name in layer_modules}
handles = []

def capture(name):
    def hook(_module, _inputs, output):
        tensor = output[0] if isinstance(output, tuple) else output
        embedding = tensor[:, 0] if name == 'final-cls' else tensor.mean(dim=1)
        captured[name].append(embedding.detach().cpu())
    return hook

for name, module in layer_modules.items():
    handles.append(module.register_forward_hook(capture(name)))

try:
    with torch.inference_mode():
        for start in range(0, len(all_iq), 32):
            iq = all_iq[start:start + 32]
            batch = torch.from_numpy(np.stack((iq.real, iq.imag), axis=1))
            model(batch.to(device=DEVICE, dtype=torch.float32))
finally:
    for handle in handles:
        handle.remove()

embeddings = {name: torch.cat(parts).numpy() for name, parts in captured.items()}
for name, values in embeddings.items():
    print(f'{name:10s}: {values.shape}')

## 4. Metric one: counterfactual embedding drift

Because every impaired example has a clean partner, we can measure cosine distance directly. A distance near zero means the direction of the representation barely changed; a larger distance means the impairment moved the example through embedding space.

Low drift is not automatically good. If an impairment destroys class information, invariance would be harmful. Read this plot alongside the class probes in the next section.

In [ ]:
def cosine_distance(left, right):
    numerator = np.sum(left * right, axis=1)
    denominator = np.linalg.norm(left, axis=1) * np.linalg.norm(right, axis=1)
    return 1 - numerator / np.clip(denominator, 1e-12, None)

layer_names = list(embeddings)
drift = np.zeros((len(VIEW_NAMES) - 1, len(layer_names)))
for layer_index, name in enumerate(layer_names):
    values = embeddings[name].reshape(len(VIEW_NAMES), len(base_iq), -1)
    for view_index in range(1, len(VIEW_NAMES)):
        drift[view_index - 1, layer_index] = cosine_distance(
            values[0], values[view_index]
        ).mean()

fig, axis = plt.subplots(figsize=(9, 4.5), constrained_layout=True)
for row, view_name in zip(drift, VIEW_NAMES[1:]):
    axis.plot(layer_names, row, marker='o', linewidth=2, label=view_name)
axis.set(title='Mean paired cosine distance through XCiT', xlabel='Representation depth', ylabel='Cosine distance')
axis.grid(alpha=0.25)
axis.legend()
plt.show()

## 5. Metric two: class and nuisance linear probes

A linear probe asks what information is easily decodable from a frozen representation. We create a stratified identity split once, then apply it to every view. This prevents a probe from training on the clean version of an identity and testing on its impaired twin.

The class probe is trained only on clean training identities and evaluated on every view of held-out identities. The nuisance probe sees all views of training identities and predicts which view produced a held-out embedding. High nuisance accuracy means the layer still exposes impairment information linearly—even if the final classifier prediction is stable.

In [ ]:
train_identity, test_identity = [], []
split_rng = np.random.default_rng(SEED + 1)
for label in range(len(CLASS_NAMES)):
    identities = identity_id[class_id == label].copy()
    split_rng.shuffle(identities)
    cut = int(0.67 * len(identities))
    train_identity.extend(identities[:cut])
    test_identity.extend(identities[cut:])
train_identity = np.asarray(train_identity)
test_identity = np.asarray(test_identity)
train_mask = np.isin(all_identity, train_identity)
test_mask = np.isin(all_identity, test_identity)
clean_train_mask = train_mask & (all_view == 0)
print(f'Train identities={len(train_identity)}, test identities={len(test_identity)}')

In [ ]:
class_accuracy = np.zeros((len(VIEW_NAMES), len(layer_names)))
nuisance_accuracy = np.zeros(len(layer_names))
for layer_index, name in enumerate(layer_names):
    values = embeddings[name]
    class_probe = make_pipeline(
        StandardScaler(), LogisticRegression(max_iter=2000, C=1.0)
    )
    class_probe.fit(values[clean_train_mask], all_class[clean_train_mask])
    for view_index in range(len(VIEW_NAMES)):
        evaluation = test_mask & (all_view == view_index)
        class_accuracy[view_index, layer_index] = accuracy_score(
            all_class[evaluation], class_probe.predict(values[evaluation])
        )

    nuisance_probe = make_pipeline(
        StandardScaler(), LogisticRegression(max_iter=2000, C=1.0)
    )
    nuisance_probe.fit(values[train_mask], all_view[train_mask])
    nuisance_accuracy[layer_index] = accuracy_score(
        all_view[test_mask], nuisance_probe.predict(values[test_mask])
    )

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), constrained_layout=True)
for row, view_name in zip(class_accuracy, VIEW_NAMES):
    axes[0].plot(layer_names, row, marker='o', label=view_name)
axes[0].axhline(1 / len(CLASS_NAMES), color='gray', linestyle='--', label='class chance')
axes[0].set(title='Clean-trained class probe', xlabel='Representation depth', ylabel='Held-out accuracy', ylim=(0, 1.05))
axes[0].legend(fontsize=8)
axes[1].plot(layer_names, nuisance_accuracy, marker='o', color='#D55E00', linewidth=2)
axes[1].axhline(1 / len(VIEW_NAMES), color='gray', linestyle='--', label='nuisance chance')
axes[1].set(title='Nuisance probe', xlabel='Representation depth', ylabel='Held-out accuracy', ylim=(0, 1.05))
axes[1].legend()
for axis in axes:
    axis.grid(alpha=0.25)
plt.show()

## 6. Visualize the final representation—carefully

PCA gives a two-dimensional projection of the final classification-token embeddings. Color encodes waveform family and marker shape encodes the impairment view. Separation or overlap in two dimensions is suggestive, not definitive: always interpret it alongside the quantitative paired distances and probes.

In [ ]:
final_values = StandardScaler().fit_transform(embeddings['final-cls'])
coordinates = PCA(n_components=2, random_state=SEED).fit_transform(final_values)
colors = ['#0072B2', '#D55E00', '#009E73']
markers = ['o', 's', '^', 'X']
fig, axis = plt.subplots(figsize=(9, 7), constrained_layout=True)
for class_index, class_name in enumerate(CLASS_NAMES):
    for view_index, view_name in enumerate(VIEW_NAMES):
        selected = (all_class == class_index) & (all_view == view_index)
        axis.scatter(
            coordinates[selected, 0], coordinates[selected, 1],
            color=colors[class_index], marker=markers[view_index],
            alpha=0.7, s=45, label=f'{class_name} / {view_name}',
        )
axis.set(title='PCA of final XCiT classification-token embeddings', xlabel='PC 1', ylabel='PC 2')
axis.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
axis.grid(alpha=0.2)
plt.show()

## 7. How to read the audit

Look for combinations, not one winning number:

- **Low paired drift + stable class-probe accuracy:** useful invariance.
- **High drift + stable class accuracy:** the representation changes, but class information survives.
- **High nuisance-probe accuracy:** impairment information remains linearly accessible. That may be harmless, useful for downstream tasks, or evidence of an unwanted shortcut.
- **Falling class accuracy under one view:** the clean decision geometry does not transfer to that impairment.

For a stronger study, replace the analytic waveforms with held-out TorchSig identities, repeat multiple seeds, bootstrap confidence intervals, tune probe regularization on a nested validation split, compare centered-kernel alignment or CKA across layers, and audit several impairment severities rather than one point.